### Day 2 Assignment: Unity Catalog Data Objects — Catalogs, Schemas, Tables, Views & UDFs

### Basic Tasks

#### 1. Create catalog and schema 

In [0]:
CREATE CATALOG IF NOT EXISTS cyntexa_dev;

CREATE SCHEMA IF NOT EXISTS cyntexa_dev.sales;

#### 2. Create Managed Table & Insert 10 Rows

In [0]:
CREATE TABLE IF NOT EXISTS cyntexa_dev.sales.orders_raw (
    order_id INT,
    customer_id STRING,
    order_date DATE,
    total_amount DECIMAL(10, 2),
    order_status STRING,
    payment_method STRING
);

In [0]:
INSERT INTO cyntexa_dev.sales.orders_raw VALUES
    (101, 'CUST-1001', '2026-08-01', 150.00, 'COMPLETED', 'Credit Card'),
    (102, 'CUST-1002', '2026-08-02', 200.50, 'PENDING',   'UPI'),
    (103, 'CUST-1003', '2026-08-02', 89.99,  'COMPLETED', 'Debit Card'),
    (104, 'CUST-1001', '2026-08-03', 450.00, 'CANCELLED', 'Credit Card'),
    (105, 'CUST-1004', '2026-08-04', 120.00, 'COMPLETED', 'Net Banking'),
    (106, 'CUST-1005', '2026-08-05', 60.00,  'FAILED',    'UPI'),
    (107, 'CUST-1002', '2026-08-06', 310.00, 'COMPLETED', 'Credit Card'),
    (108, 'CUST-1006', '2026-08-07', 95.50,  'COMPLETED', 'Debit Card'),
    (109, 'CUST-1007', '2026-08-08', 540.00, 'PENDING',   'Net Banking'),
    (110, 'CUST-1003', '2026-08-09', 215.00, 'COMPLETED', 'UPI');

#### 3. Create View for Completed Orders

In [0]:
CREATE OR REPLACE VIEW cyntexa_dev.sales.orders_view AS 
SELECT * FROM cyntexa_dev.sales.orders_raw
WHERE order_status = 'COMPLETED' 

#### 4. Explore samples.tpch (3 Exploratory Queries)

In [0]:
--Total Revenue per Customer
SELECT 
    c_custkey,
    c_name,
    sum(o_totalprice) as total_revenue
FROM samples.tpch.orders AS o
JOIN samples.tpch.customer AS c
ON o.o_custkey = c.c_custkey
GROUP BY c_custkey, c_name
ORDER BY total_revenue DESC

In [0]:
-- Top 5 nations by customer count
SELECT 
    n.n_name AS nation_name,
    COUNT(c.c_custkey) AS total_customers
FROM samples.tpch.nation n
JOIN samples.tpch.customer c ON n.n_nationkey = c.c_nationkey
GROUP BY n.n_name
ORDER BY total_customers DESC
LIMIT 5;

In [0]:
-- Top 10 orders by total revenue
SELECT 
    o_orderkey,
    SUM(o_totalprice) AS total_revenue
FROM samples.tpch.orders
GROUP BY o_orderkey
ORDER BY total_revenue DESC
LIMIT 10;
     

### Intermediate Tasks

#### 5. SQL UDF for Masking

In [0]:
CREATE OR REPLACE FUNCTION cyntexa_dev.sales.datamask(val STRING)
RETURNS STRING
RETURN CONCAT(repeat("*", length(val)-4), right(val, 4))

In [0]:
SELECT 
    order_id,
    cyntexa_dev.sales.datamask(customer_id) AS masked_customer_id,
    order_date,
    total_amount,
    order_status
FROM cyntexa_dev.sales.orders_raw;

#### 6. Create an external table

In [0]:
-- -- Create an external table pointing to external storage
-- CREATE TABLE IF NOT EXISTS cyntexa_dev.sales.orders_external (
--     order_id INT,
--     customer_id STRING,
--     order_date DATE,
--     total_amount DECIMAL(10, 2)
-- )
-- LOCATION 'dbfs:/mnt/cyntexa_storage/sales/orders_external';

-- -- Inspect both tables to compare metadata
-- DESCRIBE EXTENDED cyntexa_dev.sales.orders_raw;
-- DESCRIBE EXTENDED cyntexa_dev.sales.orders_external;

-- Creating table in Unity Catalog with file scheme dbfs is not supported

#### 7: View Joining Customers and Calculating Total Spend

In [0]:
CREATE TABLE IF NOT EXISTS cyntexa_dev.sales.customers (
    customer_id STRING,
    customer_name STRING,
    region STRING
);

In [0]:
INSERT INTO cyntexa_dev.sales.customers VALUES
    ('CUST-1001', 'Alice Johnson', 'North'),
    ('CUST-1002', 'Bob Smith', 'West'),
    ('CUST-1003', 'Charlie Brown', 'East'),
    ('CUST-1004', 'Diana Prince', 'South'),
    ('CUST-1005', 'Evan Wright', 'North'),
    ('CUST-1006', 'Fiona Gallagher', 'East'),
    ('CUST-1007', 'George Clark', 'West');

In [0]:
CREATE OR REPLACE VIEW cyntexa_dev.sales.customer_spend_view AS
SELECT
    c.customer_id,
    c.customer_name,
    c.region,
    COUNT(v.order_id) AS completed_orders_count,
    ROUND(COALESCE(SUM(v.total_amount), 0), 2) AS total_spend
FROM cyntexa_dev.sales.customers AS c 
JOIN cyntexa_dev.sales.orders_view AS v
ON c.customer_id = v.customer_id
GROUP BY c.customer_id, c.customer_name, c.region;

In [0]:
SELECT * FROM cyntexa_dev.sales.customer_spend_view 
ORDER BY total_spend DESC

### Advanced Tasks

#### 8. Three-level namespace plan

Catalog Tier (Environment Boundary)

-  cyntexa_dev   (Engineering development, experimentation, scratchpads)
-  cyntexa_stage (Pre-production integration, QA testing, staging)
-  cyntexa_prod  (Production SLA dashboards, operational reporting, ML endpoints)

Schema Tier (Business Domain / Medallion Layers within each Catalog)

-  sales          (Orders, pipeline conversions, revenue transactions)
-  marketing      (Campaign performance, leads, attribution) 
-  customer_ops   (Support tickets, customer health scores, telemetry) 
-  finance        (Invoicing, billing, cost reconciliation)

Justification:
Using Catalogs as Environment Boundaries (dev/stage/prod) guarantees physical and operational isolation so non-prod testing never impacts production workloads. Placing Schemas by Domain (sales, finance, marketing) within each catalog allows fine grained role-based access control (RBAC) and keeps cross-functional teams autonomous.


#### 9. Data Masking Strategy Document

Columns Requiring Masking: PII and sensitive financial attributes (e.g., email, phone_number, tax_id, customer_ssn, raw_payment_token).

Role Tiers & Visibility:

Tier 1: Admin & Compliance Officers
 View full, unmasked raw data for audits.
 
Tier 2: Analysts & Data Scientists 
 View dynamically masked values (e.g., jo*****@email.com or ****-1001).
 
Tier 3: External / General Users 
 Completely redacted or tokenized access.

In [0]:
-- Unity Catalog Enforcement: Enforced natively via Column Masks and Row Filters:
CREATE OR REPLACE FUNCTION cyntexa_dev.sales.mask_pii(val STRING)
RETURNS STRING
RETURN IF(IS_ACCOUNT_GROUP_MEMBER('compliance_admins'), val, CONCAT(LEFT(val, 2), '*****'));

ALTER TABLE cyntexa_dev.sales.customers 
ALTER COLUMN customer_name SET MASK cyntexa_dev.sales.mask_pii;

#### 10. Top 5 Customers by Revenue per Region (samples.tpch)

In [0]:
-- Using samples.tpch, write a query with at least one CTE and one window function to
-- produce a 'top 5 customers by revenue per region' report.

WITH customer_revenue AS (
  SELECT
    r.r_name,
    c.c_custkey,
    c.c_name,
    SUM(l.l_extendedprice * (1 - l.l_discount)) AS total_revenue,
    DENSE_RANK() OVER (PARTITION BY r.r_name ORDER BY SUM(l.l_extendedprice * (1 - l.l_discount)) DESC) AS rank
  FROM
    samples.tpch.region AS r
    JOIN samples.tpch.nation AS n 
    ON r.r_regionkey = n.n_regionkey
    JOIN samples.tpch.customer AS c 
    ON n.n_nationkey = c.c_nationkey
    JOIN samples.tpch.orders AS o
    ON c.c_custkey = o.o_custkey
    JOIN samples.tpch.lineitem AS l 
    ON o.o_orderkey = l.l_orderkey
    GROUP BY r.r_name, c.c_custkey, c.c_name
)
SELECT
  r_name,
  c_custkey,
  c_name,
  total_revenue
FROM customer_revenue
WHERE rank <= 5
ORDER BY r_name, rank;